In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

In [2]:
def phase1(path):
    data=pd.read_csv(path)
    print('INFORMATION ABOUT THE DATA')
    print(data.info())
    print("to see the number of unique values each columns hold")
    for i in data.columns:
        print(f'{i} : {data[i].nunique()}')
    print("to see that each columns carries how many null values")
    print(data.isnull().sum())
    print('total null values')
    print(data.isnull().sum().sum())
    profile=ProfileReport(data,explorative=True,title="EDA")
    profile.to_file("new_report.html")
    print('profile report of the data is generated')


In [3]:
path=r"D:\ml_rev\medical_dataset\insurance.csv"

In [4]:
phase1(path)

INFORMATION ABOUT THE DATA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None
to see the number of unique values each columns hold
age : 47
sex : 2
bmi : 548
children : 6
smoker : 2
region : 4
charges : 1337
to see that each columns carries how many null values
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
total null values
0


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 209.11it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

profile report of the data is generated


In [5]:
def tonumeric(data,x):
    data[x]=pd.to_numeric(data[x],errors='coerce')

In [6]:
def missing_val(data,x,y):
    if y=="mean":
        data[x]=data[x].fillna(data[x].mean())
    elif y=="mode":
        data[x]=data[x].fillna(data[x].mode())
    elif y=="median":
        data[x]=data[x].fillna(data[x].median())
    else:
        data[x]=data[x].fillna(0)

In [18]:
data=pd.read_csv(path)

In [20]:
def Drop_duplicates(data):
    data.drop_duplicates()

In [28]:
def encoding_col(data):
    cols=[]
    for i in data.columns:
        if data[i].dtype== 'object':
            cols.append(i)
    return cols

In [29]:
def encoding(data,Type):
    cols=encoding_col(data)
    if Type=="o":
        data=pd.get_dummies(data,dtype=int)
    else:
        from sklearn.preprocessing import LabelEncoder
        le=LabelEncoder()
        for i in cols:
            data[i]=le.fit_transform(data[i])

In [33]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

In [ ]:
def vif_handling(data,a1):
    for i in range(1,100):
        VIF=pd.DataFrame()
        VIF['VIF']=0
        cols=[]
        for i in data.columns:
            if data[i].dtype != "object":
                cols.append(i)
        VIF['FEATURES']=cols
        x=data[cols]
        vif_val=[]
        for i in range(len(cols)):
            v=vif(x.values,i)
            vif_val.append(v)
        VIF['VIF']=vif_val
        a=a1
        for i in VIF['VIF']:
            if a<i:
                a=i
        if a>10:
            x=VIF.loc[VIF['VIF']==a,'FEATURES'].iloc[0]
            data.drop(x,axis=1,inplace=True)
        else:
            break

In [39]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [40]:
def d_reduction(data,y,d,n):
    if d=="r":
        pca=PCA(n_components=n)
        data1=pca.fit_transform(data)
    else:
        lda=LinearDiscriminantAnalysis()
        data1=lda.fit_transform(data,y)

In [41]:
from imblearn.over_sampling import SMOTE

In [45]:
def outlier_handling(data,cols):
    for i in cols:
        if data[i].dtype != "object":
            q1=data[i].quantile(0.25)
            q3=data[i].quantile(0.75)
            i1=q1-(q1*1.5)
            i2=q3+(q3*1.5)
            data=data[(data[i]>=i1) & (data[i]<=i2)]

In [46]:
def oversampling(data,y):
    smote=SMOTE()
    data,y=SMOTE.fit_resample(data,y)

In [47]:
from sklearn.model_selection import train_test_split as tts
from sklearn.feature_selection import RFE

In [48]:
def splitting(data,y,s,r):
    x_train,x_test,y_train,y_test=tts(x,y,test_size=s,random_state=r)
    return x_train,x_test,y_train,y_test

In [49]:
def training(model,x_train,y_train):
    model.fit(x_train,y_train)

In [50]:
def predict(model,y_test):
    y_pred=model.predict(y_test)